### 1.3.7.4. Derivatives of Trace, Determinant, and Log-Det

$$
\frac{\partial}{\partial X}\operatorname{tr}(AX) = A^{\top},
\qquad
\frac{\partial}{\partial X}\det(X) = \det(X)\,X^{-\top},
\qquad
\frac{\partial}{\partial X}\ln\det(X) = X^{-\top} .
$$

**Explanation:**

These three matrix gradients power likelihood and barrier computations. The trace derivative is the linear building block; the [determinant](../../01_Linear_Algebra/03_Matrix/10_matrix_determinant.ipynb) derivative follows from cofactor expansion as $\det(X)\,X^{-\top}$; and dividing by $\det(X)$ gives the especially clean $\partial\ln\det(X)/\partial X = X^{-\top}$ (which is $X^{-1}$ for symmetric $X$). The log-det gradient is exactly what appears when maximizing a [Gaussian log-likelihood](../../../03_Optimization/02_Convex_Optimization/07_Statistical_Estimation/01_maximum_likelihood_as_convex_optimization.ipynb) over a covariance matrix and in the log-det barrier of interior-point methods.

**Properties:**
- For symmetric positive-definite $X$, $\partial\ln\det(X)/\partial X = X^{-1}$.
- $\ln\det$ is concave on the positive-definite cone, so $-\ln\det$ is a convex barrier.

**Numerical Example:**

For a general $2\times 2$ matrix $X = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, $\det(X) = ad - bc$ and $\ln\det(X) = \ln(ad - bc)$. The entrywise partials are

$$
\frac{\partial \ln\det}{\partial X} =
\frac{1}{ad - bc}\begin{bmatrix} d & -c \\ -b & a \end{bmatrix} = X^{-\top} .
$$

For $\det$ itself the same pattern without the $1/\det$ factor gives $\begin{bmatrix} d & -c \\ -b & a \end{bmatrix} = \det(X)\,X^{-\top}$, since $X^{-1} = \tfrac{1}{\det}\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$.

In [1]:
import sympy as sp

a, b, c, d = sp.symbols("a b c d")
X = sp.Matrix([[a, b], [c, d]])

determinant = X.det()
log_det = sp.log(determinant)

grad_det = X.applyfunc(lambda entry: sp.diff(determinant, entry))
grad_log_det = X.applyfunc(lambda entry: sp.diff(log_det, entry))

print("∂det(X)/∂X        ="); sp.pprint(grad_det)
print("equals det(X)·X⁻ᵀ :", sp.simplify(grad_det - determinant * X.inv().T) == sp.zeros(2, 2))
print("∂ln det(X)/∂X     ="); sp.pprint(sp.simplify(grad_log_det))
print("equals X⁻ᵀ        :", sp.simplify(grad_log_det - X.inv().T) == sp.zeros(2, 2))

∂det(X)/∂X        =
⎡d   -c⎤
⎢      ⎥
⎣-b  a ⎦
equals det(X)·X⁻ᵀ : True
∂ln det(X)/∂X     =
⎡    d         -c    ⎤
⎢─────────  ─────────⎥
⎢a⋅d - b⋅c  a⋅d - b⋅c⎥
⎢                    ⎥
⎢   -b          a    ⎥
⎢─────────  ─────────⎥
⎣a⋅d - b⋅c  a⋅d - b⋅c⎦
equals X⁻ᵀ        : True


**Equivalent CasADi implementation:**

CasADi differentiates $\ln\det(X)$ with respect to the vectorized matrix and matches $X^{-\top}$ numerically.

In [2]:
import casadi as ca

X = ca.SX.sym("X", 2, 2)
log_det = ca.log(ca.det(X))
grad_log_det = ca.reshape(ca.gradient(log_det, ca.vec(X)), 2, 2)
grad_function = ca.Function("grad_log_det", [X], [grad_log_det])

sample = ca.DM([[4, 1], [1, 3]])
print("∂ln det/∂X at sample =", grad_function(sample))
print("X⁻ᵀ at sample        =", ca.inv(sample).T)

∂ln det/∂X at sample = 
[[0.272727, -0.0909091], 
 [-0.0909091, 0.363636]]
X⁻ᵀ at sample        = 
[[0.272727, -0.0909091], 
 [-0.0909091, 0.363636]]


**References:**

[📗 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*, Appendix A.4.](https://web.stanford.edu/~boyd/cvxbook/)  
[📘 Bertsekas, D. P. (1999). *Nonlinear Programming*, Appendix A.5.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Gradients of Linear and Quadratic Forms](./03_gradients_of_linear_and_quadratic_forms.ipynb) | [Next: Matrix Chain Rule ➡️](./05_matrix_chain_rule.ipynb)